In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn import metrics

In [2]:
df = pd.read_csv('sampled_data.csv')
df.head()

,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,443,6,14/02/2018 12:41:31,116087899,21,19,750,5575,381,0,...,20,20977.833333,37055.338921,138644,10179,9.653011e+06,1.202897e+06,10005066,5833596,Benign
1,21,6,14/02/2018 11:13:32,1,1,1,0,0,0,0,...,40,0.000000,0.000000,0,0,0.000000e+00,0.000000e+00,0,0,FTP-BruteForce
2,3389,6,14/02/2018 02:34:56,1206036,8,7,1148,1581,677,0,...,20,0.000000,0.000000,0,0,0.000000e+00,0.000000e+00,0,0,Benign
3,53,17,14/02/2018 03:56:36,1273,1,1,32,76,32,32,...,8,0.000000,0.000000,0,0,0.000000e+00,0.000000e+00,0,0,Benign
4,21,6,14/02/2018 11:01:12,2,1,1,0,0,0,0,...,40,0.000000,0.000000,0,0,0.000000e+00,0.000000e+00,0,0,FTP-BruteForce


In [3]:
df = df.drop("Timestamp", axis=1)
df["Label"] = df["Label"].map({"Benign": 0, "FTP-BruteForce": 1, "SSH-Bruteforce": 2})
df.drop(columns=["Flow Byts/s"], inplace=True)

In [4]:
df_cleaned = df[(df >= 0).all(axis=1)]

In [5]:
# Check for infinite values in features
inf_check = df_cleaned.isin([np.inf, -np.inf]).any()
columns_with_inf = inf_check[inf_check].index.tolist()
print("Columns with infinities:", columns_with_inf)

Columns with infinities: ['Flow Pkts/s']


In [6]:
# Replace inf/-inf with NaN and drop rows
df_cleaned = df_cleaned.replace([np.inf, -np.inf], np.nan).dropna()
print("Remaining infinities in df_cleaned:", np.isinf(df_cleaned).any().any())

Remaining infinities in df_cleaned: False


In [7]:
scaler = StandardScaler()
df_cleaned_scaled = scaler.fit_transform(df_cleaned)

In [8]:
x = df_cleaned.drop(['Label'], axis=1)
y = df_cleaned['Label']
print(y.unique())

[0 1 2]


In [9]:
X_train, X_test, y_train, y_test = train_test_split(x, y,test_size = 0.1)

In [10]:
X_train.nunique()

Dst Port          1459
Protocol             1
Flow Duration    15389
Tot Fwd Pkts       140
Tot Bwd Pkts       192
                 ...  
Active Min        2628
Idle Mean         2928
Idle Std          2306
Idle Max          2826
Idle Min          2878
Length: 77, dtype: int64

In [11]:
Classifier_accuracy = []

In [12]:
knn_clf = KNeighborsClassifier()
knn_clf.fit(X_train, y_train)
y_pred = knn_clf.predict(X_test)
accuracy = metrics.accuracy_score(y_test, y_pred)
Classifier_accuracy.append(accuracy*100)
print("Accuracy of KNN Classifier : %.2f" % (accuracy*100))

Accuracy of KNN Classifier : 99.79


The base line of KNN accived a very high accuracy of 99.81. 

In [13]:
from sklearn.metrics import classification_report

# Define class names as strings in the order of numeric labels
labels = ['0', '1', '2']  # Replace with your actual class names

# Generate the report
print(classification_report(y_test, y_pred, target_names=labels))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1439
           1       1.00      1.00      1.00       986
           2       0.99      1.00      1.00       977

    accuracy                           1.00      3402
   macro avg       1.00      1.00      1.00      3402
weighted avg       1.00      1.00      1.00      3402

